# Scientific Programming with Python 
## (Winter 2025/26)

# Session 07: Pandas II

* Some more python: decorators
* Pandas II

# Decorators

* a decorator is a design pattern that allows you to modify the functionality of a function by wrapping it in another function.
* The outer function is called the **decorator**, which takes the original function as an argument and returns a modified version of it.

## Pass Function as Argument

* function are, like everything else in Python, objects
* functions can be assigned to variables
* functions can be passed as arguments to other functions
* they can be the return value of (another) function

In [ ]:
def add(x, y):
    return x + y

def calculate(func, x, y):
    return func(x, y)

result = calculate(add, 4, 6)
print(result)

In [ ]:
def say_hello(name):
    return f"Hello {name}"

def be_awesome(name):
    return f"Yo {name}, together we're the awesomest!"

def greet_bob(greeter_func):
    return greeter_func("Bob")

print(greet_bob(say_hello))
print(greet_bob(be_awesome))

## Nested functions

We can include one function inside another, known as a nested function.

In [ ]:
def parent():
    print("Printing from parent()")

    def first_child():
        print("Printing from first_child()")

    def second_child():
        print("Printing from second_child()")

    second_child()
    first_child()

In [ ]:
parent()

In [ ]:
def outer(x):
    def inner(y):
        return x + y
    return inner

add_five = outer(5)
result = add_five(6)
print(result)

In [ ]:
def greeting(name):
    def hello():
        return "Hello, " + name + "!"
    return hello

greet = greeting("World")
print(greet())

In [ ]:
def parent(num):
    def first_child():
        return "Hi, I'm Ann"

    def second_child():
        return "Call me Bob"

    if num == 1:
        return first_child
    else:
        return second_child

In [ ]:
first = parent(1)
second = parent(2)

In [ ]:
first()

In [ ]:
second()

## Python Decorators

Decorators are functions that wrap another function.

In [ ]:
def decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()
        print("Something is happening after the function is called.")
    return wrapper

def say_whee():
    print("Whee!")

decorated_say_whee = decorator(say_whee)

In [ ]:
decorated_say_whee()

In [ ]:
from datetime import datetime

def not_during_the_night(func):
    def wrapper():
        if 7 <= datetime.now().hour < 22:
            func()
        else:
            pass  # Hush, the neighbors are asleep
    return wrapper

def say_whee():
    print("Whee!")

say_whee = not_during_the_night(say_whee)

### The @-syntax

Python provides syntactic sugar for function decoration: the @-notation

```python
@decorator
def my_function():
    ...
```
* `decorator` has to be the name of a decorator function
* `my_function` becomes the name of the decorated function (not the original one)
* essentially equivalent to saying `my_function = decorator(my_function)`

In [ ]:
def decorator(func):
    def wrapper():
        print("Something is happening before the function is called.")
        func()
        print("Something is happening after the function is called.")
    return wrapper

@decorator
def say_whee():
    print("Whee!")
    

In [ ]:
say_whee()

**Exercise:**
1. define a simple decorator `do_twice` that runs the decorated function twice.
2. define a simple decorator `separate` that separates the output of the decorated function from text before and after by some lines.

In [ ]:
def do_twice(func):
    ### BEGIN SOLUTION
    def wrapper_do_twice():
        func()
        func()
    return wrapper_do_twice
    ### END SOLUTION

@do_twice
def say_whee():
    print("Whee!")

say_whee()

In [ ]:
def separate(func):
    ### BEGIN SOLUTION
    def wrapper():
        print("-"*80)
        func()
        print("-"*80)
    return wrapper
    ### END SOLUTION

@separate
def say_whee():
    print("Whee!")

say_whee()

## Decorating functions with parameters

In [ ]:
def divide(a, b):
    return a/b

In [ ]:
def smart_divide(func):
    def inner(a, b):
        print("I am going to divide", a, "and", b)
        if b == 0:
            print("Whoops! cannot divide")
            return

        return func(a, b)
    return inner

In [ ]:
@smart_divide
def divide(a, b):
    print(a/b)

divide(2,5)

divide(2,0)

In [ ]:
def do_twice(func):
    def wrapper_do_twice(name):
        func(name)
        func(name)
    return wrapper_do_twice

@do_twice
def greet(name):
    print(f"Hello {name}")

greet("Bob")

Problem:
* the decorator needs to know what parameters the decorated function expects
* this drastically limits the reusability of a decorator

Solution: use the generic argumnts notation
- `*args`: a tuple of all positional arguments
- `**kwargs`: a dict with all keyword arguments

Pass those arguments to the function:

In [ ]:
def do_twice(func):
    def wrapper_do_twice(*args, **kwargs):
        func(*args, **kwargs)
        func(*args, **kwargs)
    return wrapper_do_twice

In [ ]:
@do_twice
def greet(name, salut):
    print(f"{salut} {name}")

greet("Bob", "Hello")

## Returning values from decorated functions

If the wrapped function returns a value, this value should not be ignored:

In [ ]:
@do_twice
def return_greeting(name):
    print("Creating greeting")
    return f"Hi {name}"

return_greeting('Bob')

The wrapper function can pass back that return value to the caller:

In [ ]:
def do_twice(func):
    def wrapper_do_twice(*args, **kwargs):
        func(*args, **kwargs)
        return func(*args, **kwargs)
    return wrapper_do_twice

In [ ]:
@do_twice
def return_greeting(name):
    print("Creating greeting")
    return f"Hi {name}"

return_greeting('Bob')

**Exercise:** adapt the implementation of `do_twice` so that it returns a both return values instead of only the last one.

In [ ]:
def do_twice(func):
    ### BEGIN SOLUTION
    def wrapper_do_twice(*args, **kwargs):
        return (func(*args, **kwargs),
                func(*args, **kwargs))
    return wrapper_do_twice
    ### END SOLUTION

@do_twice
def return_greeting(name):
    print("Creating greeting")
    return f"Hi {name}"

return_greeting('Bob')

In [ ]:
import time

@do_twice
def perf_counter():
    return time.perf_counter()

perf_counter()

Another example: a debug function outputting arguments and return value:

In [ ]:
def debug(func):
    """Print the function signature and return value"""
    def wrapper_debug(*args, **kwargs):
        args_repr = [repr(a) for a in args]
        kwargs_repr = [f"{k}={repr(v)}" for k, v in kwargs.items()]
        signature = ", ".join(args_repr + kwargs_repr)
        print(f"Calling {func.__name__}({signature})")
        value = func(*args, **kwargs)
        print(f"{func.__name__}() returned {repr(value)}")
        return value
    return wrapper_debug

In [ ]:
@debug
def make_greeting(name, age=None):
    if age is None:
        return f"Howdy {name}!"
    else:
        return f"Whoa {name}! {age} already, you're growing up!"

In [ ]:
make_greeting("Benjamin")

In [ ]:
make_greeting("Juan", age=114)

In [ ]:
import math

math.factorial = debug(math.factorial)

def approximate_e(terms=18):
    return sum(1 / math.factorial(n) for n in range(terms))

approximate_e(terms=15)

***Exercise:*** write a decorator `timer` that outputs the runtime of a decorated function.  Take into account that
- the decorated function may expect arguments
- the decorated function may return a value

Hint: you may use `time.perf_counter()` to get a good timer value.  Compute the difference of that timer after and before the function call to estimate the runtime of the function.

In [ ]:
import time

def timer(func):
    """Print the runtime of the decorated function"""

    ### BEGIN SOLUTION
    def wrapper_timer(*args, **kwargs):
        start_time = time.perf_counter()
        value = func(*args, **kwargs)
        end_time = time.perf_counter()
        run_time = end_time - start_time
        print(f"Finished {func.__name__}() in {run_time:.4f} secs")
        return value
    return wrapper_timer
    ### END SOLUTION

@timer
def waste_some_time(num_times):
    for _ in range(num_times):
        sum([number**2 for number in range(1_000_000)])

waste_some_time(10)

## Chaining decorators

In [ ]:
def star(func):
    def inner(*args, **kwargs):
        print("*" * 15)
        func(*args, **kwargs)
        print("*" * 15)
    return inner

def percent(func):
    def inner(*args, **kwargs):
        print("%" * 15)
        func(*args, **kwargs)
        print("%" * 15)
    return inner

@star
@percent
def printer(msg):
    print(msg)

printer("Hello")

A decorator may also store values between calls. Example:

In [ ]:
def count_calls(func):
    def wrapper_count_calls(*args, **kwargs):
        wrapper_count_calls.num_calls += 1
        print(f"Call {wrapper_count_calls.num_calls} of {func.__name__}()")
        return func(*args, **kwargs)
    wrapper_count_calls.num_calls = 0
    return wrapper_count_calls

@count_calls
def fibonacci(num):
    if num < 2:
        return num
    return fibonacci(num - 1) + fibonacci(num - 2)

fibonacci(6)

This mechanism may be used to cache computations:

In [ ]:
def cache(func):
    """Keep a cache of previous function calls"""
    def wrapper_cache(*args, **kwargs):
        cache_key = args + tuple(kwargs.items())
        if cache_key not in wrapper_cache.cache:
            wrapper_cache.cache[cache_key] = func(*args, **kwargs)
        return wrapper_cache.cache[cache_key]
    wrapper_cache.cache = {}
    return wrapper_cache

@cache
@count_calls
def fibonacci(num):
    if num < 2:
        return num
    return fibonacci(num - 1) + fibonacci(num - 2)

In [ ]:
fibonacci(10)

## Decorators with arguments

It would be nice to pass arguments to your decorator.  For example, `@do_twice` cound be generalized to `repeat(n)`.

```python
@repeat(4)
def myfunc(*args):
    ...
```
This will be translated into

```python
myfunc = repeat(4)(myfunc)
```

In other words, we need a function that returns a decorator (function):
1. `repeat(4)` return a decorator (`decorator_repeat`)
2. `decorator_repeat` takes a function (here `myfunc` as argument) and wraps it into another function `wrapper_repeat`
3. `wrapper_repaet` will then replace `myfunc`

In [ ]:
def repeat(num_times):
    def decorator_repeat(func):
        def wrapper_repeat(*args, **kwargs):
            for _ in range(num_times):
                value = func(*args, **kwargs)
            return value
        return wrapper_repeat
    return decorator_repeat

In [ ]:
@repeat(num_times=4)
def greet(name):
    print(f"Hello {name}")

greet('Bob')

**Exercises:** 

(1) adapt the implementation of `repeat` so that it returns a tuple of all return values instead of only the last one.
(2) allow the `seperate` decurator to take an argument, specifying a symbol that should be used for separations

In [ ]:
def repeat(num_times):
    ### BEGIN SOLUTION
    def decorator_repeat(func):
        def wrapper_repeat(*args, **kwargs):
            return tuple(func(*args, **kwargs) for _ in range(num_times))
        return wrapper_repeat
    return decorator_repeat
    ### END SOLUTION

import time

@repeat(4)
def perf_counter():
    return time.perf_counter()

perf_counter()

In [ ]:
def separate(symbol):
    ### BEGIN SOLUTION
    def decorator(func):
        def wrapper():
            print(symbol*80)
            func()
            print(symbol*80)
        return wrapper
    return decorator
    ### END SOLUTION

@separate('#')
def say_whee():
    print("Whee!")

say_whee()

## Decorating classes

Decorators can also be applied to classes:

```python

@decorator
class MyClass:
    ...
```

This is roughly equivalent to saying `MyClass = decorator(MyClass)`.  The class decorator can adapt the class definition:
* adding properties (getters and setters)
* adding methods
* adapt functionality


In [ ]:
def singleton(cls):
    """Make a class a Singleton class (only one instance)"""
    def wrapper_singleton(*args, **kwargs):
        if wrapper_singleton.instance is None:
            wrapper_singleton.instance = cls(*args, **kwargs)
        return wrapper_singleton.instance
    wrapper_singleton.instance = None
    return wrapper_singleton

In [ ]:
@singleton
class TheOne:
    pass

first_one = TheOne()
another_one = TheOne()

first_one is another_one

Example: Dataclasses

* a [dataclass](https://docs.python.org/3/library/dataclasses.html) is a class typically containing mainly data (although there aren’t really any restrictions)
* allows for immutable data
* allow for ordering of objects

In [ ]:
from dataclasses import dataclass

@dataclass
class PlayingCard:
    rank: str
    suit: str

# initializer arguments
queen_of_hearts = DataClassCard('Q', 'Hearts')

## Predefined decorators

Python provides several decorators for standard tasks
* `@classmethod`: a class method not connected to a particular instance of that class
* `@staticmethod`: a class method not connected to the class 
* `@property`: customize getters and setters for class attributes
* `@atexit.register`: register an exit handler
* `@enum.unique`: unique values in enumeration
* `@singledispatch`: allow to have different implementation of a function, switch based on first argument
* ...

## References:
* RealPython: [Primer on Python Decorators](https://realpython.com/primer-on-python-decorators/)

## Recap: The Pandas `Series` object

Two views:
* pandas `Series` as a generalized (one-dimensional) array
* pandas `Series` as a specialized dictionary

The Series-as-dictionary analogy can be made even more clear by constructing a Series object directly from a Python dictionary:

In [ ]:
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135}
population = pd.Series(population_dict)
population

In [ ]:
import numpy as np
import pandas as pd

## The Pandas DataFrame Object

The next fundamental structure in Pandas is the DataFrame. Like the Series object, it can be thought of either as a generalization of a NumPy array, or as a specialization of a Python dictionary. We'll now take a look at each of these perspectives.

### DataFrame as a generalized NumPy array

If a Series is an analog of a one-dimensional array with flexible indices, a DataFrame is an analog of a two-dimensional array with both flexible row indices and flexible column names. Just as you might think of a two-dimensional array as an ordered sequence of aligned one-dimensional columns, you can think of a DataFrame as a sequence of aligned Series objects. Here, by "aligned" we mean that they share the same index.



To demonstrate this, let's first construct a new Series listing the area of each of the five states discussed in the previous section:

In [ ]:
area_dict = {'California': 423967, 'Texas': 695662, 'New York': 141297,
             'Florida': 170312, 'Illinois': 149995}
area = pd.Series(area_dict)
area

In [ ]:
population_dict = {'California': 38332521,
                   'Texas': 26448193,
                   'New York': 19651127,
                   'Florida': 19552860,
                   'Illinois': 12882135}
population = pd.Series(population_dict)
population

Now that we have this along with the population Series from before, we can use a dictionary to construct a single two-dimensional object containing this information:

In [ ]:
states = pd.DataFrame({'population': population,
                       'area': area,
                       'country': 'USA'})
states

In [ ]:
print(states.dtypes)

This looks like a generalized dictionary! The keys are the names of the state, and the values are like a list [area, country, population]

In [ ]:
states.sort_values(by="population", ascending=False)

In [ ]:
states['population'], type(states['population'])

In [ ]:
states["population"].idxmax() #figures out the "key(s)"(indices) of the DataFrame where "population" has its max

In [ ]:
states.loc[states["population"].idxmax()] #returnes the series at the given index

In [ ]:
states.max() #returns the max for every individual column!

In [ ]:
states['California']  # Error: indices to DataFrames refer to columns, not rows!

In [ ]:
states.loc['California']

In [ ]:
states.index

In [ ]:
states.columns

In [ ]:
states.values

In [ ]:
type(states.values)

Thus the DataFrame can be thought of as a generalization of a two-dimensional NumPy array, where both the rows and columns have a generalized index for accessing the data.

### DataFrame as specialized dictionary

Similarly, we can also think of a DataFrame as a specialization of a dictionary. Where a dictionary maps a key to a value, a DataFrame maps a column name to a Series of column data. For example, asking for the 'area' attribute returns the Series object containing the areas we saw earlier:

In [ ]:
states["area"]
# note that indexing a DataFrame with square brackets gets the *column*!

In [ ]:
type(states["area"])

### Constructing DataFrame objects

A Pandas DataFrame can be constructed in a variety of ways:

#### From a single Series object

A DataFrame is a collection of Series objects, and a single-column DataFrame can be constructed from a single Series:

In [ ]:
population

In [ ]:
pd.DataFrame(population, columns=['population'])

#### From multiple Series

In [ ]:
s1 = pd.Series(['100', '200', 'python', '300.12', '400'])
s2 = pd.Series(['10', '20', 'php', '30.12', '40'])
df = pd.concat([s1, s2], axis='columns')
df

Note that many functions in pandas take the `axis` argument - In which case you can select between 0/`'index'` and  1/`'columns'`. If you want to be explicit, I recommend using the string-version!

#### From a list of dicts 

Any list of dictionaries can be made into a DataFrame. We'll use a simple list comprehension to create some data. Even if some keys in the dictionary are missing, Pandas will fill them in with NaN (i.e., "not a number") values:

In [ ]:
df = pd.DataFrame([{'a': 1, 'b': 2}, {'b': 3, 'c': 4}], index=["first_dict", "second_dict"])
df

As every single column must have a consistent dtype and np.NaN is a float, some of the numbers get coerced into floats:

In [ ]:
df['a']

In [ ]:
df['b']

In [ ]:
type(np.nan)

In [ ]:
df.dtypes

If we wanted to get the rows, pandas would need to coerce the numbers explicitly: 

In [ ]:
df

In [ ]:
df.loc['first_dict']

#### From a two-dimensional NumPy array

Given a two-dimensional array of data, we can create a DataFrame with any specified column and index names. If omitted, an integer index will be used for each:


In [ ]:
dates = pd.date_range('20130101', periods=6)
df = pd.DataFrame(np.random.randn(6,4), index=dates, columns=list('ABCD'))
df

## The Pandas Index Object

We have seen here that both the Series and DataFrame objects contain an explicit index that lets you reference and modify data. This Index object is an interesting structure in itself, and it can be thought of as an immutable array:

In [ ]:
ind = pd.Index([2, 3, 5, 7, 11])
ind

In [ ]:
ind[0] = 1  # error: indices are immutable

In [ ]:
sr = pd.Series(0, index=ind)
sr

Index objects have a name:

In [ ]:
ind.names = ['indexx']
ind

In [ ]:
sr = pd.Series(np.zeros_like(ind), index=ind)
sr

In [ ]:
df = pd.DataFrame(np.zeros_like(ind), index=ind, columns=['first'])
df

In [ ]:
df.index.names = [None]
df

Index objects also have many of the attributes familiar from NumPy arrays:

In [ ]:
ind.size, ind.shape, ind.ndim, ind.dtype

# Data Indexing, Selection, and Assignment


From the numpy lecture, we already know about indexing, slicing, masking, and fancy indexing:

In [ ]:
a = np.arange(16).reshape(4,4)
a

In [ ]:
a[:, [1, 3]][a[:, [1, 3]] % 3 == 0]
# Takes those values of the second and fourth column that are divisible by 3

Here we'll look at similar means of accessing and modifying values in Pandas Series and DataFrame objects. The corresponding patterns in Pandas are very similar to those of numpy, though there are a few quirks to be aware of.

We'll start with the simple case of the one-dimensional Series object, and then move on to the slightly more complicated two-dimensional DataFrame object.

## Data Selection in Series

As we saw in the previous section, a Series object acts in many ways like a one-dimensional NumPy array, and in many ways like a standard Python dictionary. If we keep these two overlapping analogies in mind, it will help us to understand the patterns of data indexing and selection in these arrays.

### Series as dictionary

Like a dictionary, the Series object provides a mapping from a collection of keys to a collection of values, which means most of the corresponding functions work just as well for them:

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=['a', 'b', 'c', 'd'])
data

In [ ]:
data.__contains__('b')

In [ ]:
'b' in data

In [ ]:
np.array_equal(data.keys(), data.index)

In [ ]:
data

In [ ]:
list(data.items())

In [ ]:
data['e'] = 1.25
data

### Series as one-dimensional array

Series builds on this dictionary-like interface and provides array-style item selection via the same basic mechanisms as NumPy arrays – that is, slices, masking, and fancy indexing. Examples of these are as follows:

In [ ]:
# slicing by explicit index
data['a':'c']

In [ ]:
# slicing by implicit integer index
data[0:2] 
# Note that when slicing with an explicit index (i.e., data['a':'c']), the final index is included in the slice, 
# while when slicing with an implicit index (i.e., data[0:2]), the final index is excluded from the slice.

In [ ]:
(data > 0.3) & (data < 0.8)

In [ ]:
# masking
data[(data > 0.3) & (data < 0.8)]

In [ ]:
# fancy indexing
data[['a', 'e']]

In [ ]:
data = pd.Series([0.25, 0.5, 0.75, 1.0],
                 index=[1, 2, 3, 4])
data

In [ ]:
data[1:3]

**If your Series has an explicit integer index, an indexing operation such as data[1] will use the explicit indices, while a slicing operation like data[1:3] will use the implicit Python-style index.**

In [ ]:
data = pd.Series(['a', 'b', 'c'], index=[1, 5, 3])
data

In [ ]:
# explicit index when indexing
data[1]

In [ ]:
# implicit index when slicing
data[1:3]

The **loc** attribute allows indexing and slicing that *always* references the explicit index:

In [ ]:
data.loc[1]

In [ ]:
data.loc[1:3]

Note that `loc` may or may not throw Index-Errors when slicing:

In [ ]:
data = pd.Series(['a', 'b', 'c'], index=[1, 5, 3])
data

In [ ]:
data.loc['a':'z']

In [ ]:
data.loc[3:10]

In [ ]:
data = pd.Series(['a', 'b', 'c'], index=[1, 3, 5])
data.loc[3:10]

The **iloc** attribute allows indexing and slicing that always references the implicit Python-style index:

In [ ]:
data.iloc[1]

In [ ]:
data.iloc[1:3]

Please, save yourself the pain and be always explicit about what you do -- always use ``.loc`` and ``.iloc``

### Addendum: Reindexing

In [ ]:
s = pd.Series([1, 2, 3])
s

In [ ]:
s.loc[[1, 2, 3]] # error:

instead, you should go for [`reindex`][1], which conforms the Series/DF to new index with optional filling logic.

[1]: https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.reindex.html

In [ ]:
s.reindex([1, 2, 3])

## Data Selection in DataFrame

Recall that a DataFrame acts in many ways like a two-dimensional or structured array, and in other ways like a dictionary of Series structures sharing the same index. These analogies can be helpful to keep in mind as we explore data selection within this structure.

In [ ]:
area = pd.Series({'California': 423967, 'Texas': 695662,
                  'New York': 141297, 'Florida': 170312,
                  'Illinois': 149995})
pop = pd.Series({'California': 38332521, 'Texas': 26448193,
                 'New York': 19651127, 'Florida': 19552860,
                 'Illinois': 12882135})
data = pd.DataFrame({'area':area, 'pop':pop, 'Country':'USA'})
data

Note that if we index a DataFrame, we index the **column**!!

In [ ]:
# Dictionary-style indexing results in a Series....
print(type(data["area"]))
data["area"]

In [ ]:
# We can also dereference, though it leads to side-effects if that's actually also a method...
data.area

In [ ]:
type(data.values)

With this picture in mind, many familiar array-like observations can be done on the DataFrame itself. For example, we can transpose the full DataFrame to swap rows and columns:

In [ ]:
data.T

For array-style indexing, Pandas again uses the loc and iloc indexers mentioned earlier. Using the iloc indexer, we can index the underlying array as if it is a simple NumPy array (using the implicit Python-style index), **but the DataFrame index and column labels are maintained in the result**:

In [ ]:
#indexing the underlying numpy-array...
data.values[:3, :2]

In [ ]:
data.iloc[:3, :2]

In [ ]:
data

In [ ]:
data.loc[:'Illinois', :'pop']

In [ ]:
data.loc[:,['area','pop']]

So, this is how we get a row!

In [ ]:
data.loc["California", :]

In [ ]:
# adding a new column.. (vectorized calculations!)
data['density'] = data['pop'] / data['area']
# we can combine masking with fancy indexing
data.loc[data.density > 100, ['pop', 'density']]

If you want to combine explicit and implicit indexing, you have to chain them:

In [ ]:
data.iloc[1:4].loc[:, ['pop', 'density']]

**While indexing refers to columns, slicing refers to rows:**

In [ ]:
data['area']

In [ ]:
data['Florida':'Illinois']

Again, rather be explicit about your indexing to save yourself from a lot of confusion.

In [ ]:
data['area':'pop']  # Error

In [ ]:
data.loc[:, 'area':'pop']

Fast access to a single member using **at**

In [ ]:
%%timeit
data.loc['Florida', 'pop']

In [ ]:
%%timeit
data.at['Florida', 'pop']

In [ ]:
data

|                        |              |                                          |                          |
|------------------------|--------------|------------------------------------------|--------------------------|
| **direct []-access**   |              |                                          |                          |
| One argument,   single | Column       | data['area']                             |                          |
| One argument,   slice  | Row          | data['Florida': 'Illinois']              | slice-top is included    |
| Both arguments         | only MultiInd| -                                        |                          |
| **.loc[]**             |              |                                          |                          |
| One argument,   single | Row          | data.loc['Florida']                      |                          |
| One argument,   slice  | Row          | data.loc['Florida': 'Illinois']          | slice-top  is included   |
| Both arguments, both   | Row, Column  | data.loc['Florida': 'Illinois', 'area']  | slice-top  is included   |
| **.iloc[]**            |              |                                          |                          |
| One argument,   single | Row          | data.iloc[0]                             |                          |
| One argument,   slice  | Row          | data.iloc[0, 2]                          | slice-top  is excluded   |
| Both arguments, both   | Row, Column  | data.iloc[0: 2, 0:3]                     | slice-top  is excluded   |

**Renaming indices** 

In [ ]:
df = pd.DataFrame({"a": [1, 2, 3, 4], "b": [2, 5, 7, 8]})
df = df.rename(mapper={'b': 'c'}, axis='columns')
df

### Boolean Indexing

In [ ]:
dates = pd.date_range('20130101', periods=6)
df = pd.DataFrame(np.random.randn(6,4), index=dates, columns=list('ABCD'))
df['E'] = ["one", "two", "three"] * 2
df

In [ ]:
df['A'] > 0.5

In [ ]:
df[df['A'] > 0]

In [ ]:
#alternate syntax: `query` - see https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.query.html
df.query('A > 0')

In [ ]:
df['E'].isin(['one','two'])

In [ ]:
df[df['E'].isin(['one','two'])] = np.nan
df

In [ ]:
pd.isna(df)

In [ ]:
pd.isna(df).any(axis=1)

In [ ]:
df[~df.isna().any(axis=1)]

In [ ]:
df.dropna(how="any")

## Data assignment

In [ ]:
df = pd.DataFrame({'temp_c': [17.0, 25.0]},
                  index=['Portland', 'Berkeley'])
df

### Assigning columns

In [ ]:
df['country'] = 'USA'
df

In [ ]:
df['temp_c'] <= 18

In [ ]:
df['too_cold'] = df['temp_c'] <= 18
df

These however work in-place. To assign to a new dataframe, use `assign`:

In [ ]:
df = pd.DataFrame({'temp_c': [17.0, 25.0]},
                  index=['Portland', 'Berkeley'])
df

In [ ]:
df2 = df.assign(temp_f=lambda x: x.temp_c * 9 / 5 + 32)
df2

In [ ]:
#vectorized version:
df.assign(temp_f=df['temp_c'] * 9 / 5 + 32)

In [ ]:
#multiple assignments simulatenously:
df.assign(temp_f=lambda x: x['temp_c'] * 9 / 5 + 32,
          temp_k=lambda x: (x['temp_f'] +  459.67) * 5 / 9)

In [ ]:
df

### Row assignment

In [ ]:
df.loc['Berkeley', 'temp_c'] = 26.0
df

In [ ]:
type(df.loc['Portland'])

In [ ]:
df.loc['Portland'] = pd.Series({'temp_c': 99})
df

In [ ]:
df.loc['Osnabrück', 'temp_c'] = 18
df

In [ ]:
df = pd.concat([df, df])
df

In [ ]:
df.loc['Osnabrück', 'temp_c'] = 25
df

In [ ]:
df.loc['Osnabrück'] = pd.Series({'temp_c': 99})
df

In [ ]:
type(df.loc['Osnabrück'])

In [ ]:
np.where(df.index == 'Osnabrück')

In [ ]:
df.iloc[np.where(df.index == 'Osnabrück')[0][0]]

In [ ]:
df.iloc[np.where(df.index == 'Osnabrück')[0][0]] = pd.Series({'temp_c': 99})
df

## Pandas indexing

### Multi-Indexing

While Pandas does provide objects that natively handle three-dimensional and four-dimensional data, a far more common pattern in practice is to make use of `hierarchical indexing` (also known as `multi-indexing`) to incorporate multiple index levels within a single index. In this way, higher-dimensional data can be compactly represented within the familiar one-dimensional Series and two-dimensional DataFrame objects.

In [ ]:
index = [('California', 2000), ('California', 2010),
         ('New York', 2000), ('New York', 2010),
         ('Texas', 2000), ('Texas', 2010)]
populations = [33871648, 37253956,
               18976457, 19378102,
               20851820, 25145561]
pop = pd.Series(populations, index=index)
pop

In [ ]:
index = pd.MultiIndex.from_tuples(index)
index

In [ ]:
index.names = ['state', 'year']

In [ ]:
pop = pop.reindex(index)
pop

In [ ]:
pop['California', 2000], pop['California', 2010]

In [ ]:
pop.iloc[0]

In [ ]:
pop.iloc[1]

### MultiIndex as extra dimension: stack() and unstack()

You might notice something else here: we could easily have stored the same data using a simple ``DataFrame`` with index and column labels.
In fact, Pandas is built with this equivalence in mind. The ``unstack()`` method will quickly convert a multiply indexed ``Series`` into a conventionally indexed ``DataFrame``:

In [ ]:
pop.unstack()

In [ ]:
index.names = [None, None]
pop = pop.reindex(index)
pop

In [ ]:
pop.unstack()

In [ ]:
pop.index.names = [None, None]
pop.unstack().T

In [ ]:
popdf = pop.unstack(level=0)
popdf

In [ ]:
popdf.stack()

### Index setting and resetting

Another way to rearrange hierarchical data is to turn the index labels into columns; this can be accomplished with the ``reset_index`` method.
Calling this on the population dictionary will result in a ``DataFrame`` with a *state* and *year* column holding the information that was formerly in the index.
For clarity, we can optionally specify the name of the data for the column representation:

In [ ]:
pop

In [ ]:
pop.index.names = ['state', 'year']
print(type(pop))
pop

In [ ]:
pop_flat = pop.reset_index(name='population')
print(type(pop_flat))
pop_flat

Often when working with data in the real world, the raw input data looks like this and it's useful to build a ``MultiIndex`` from the column values.
This can be done with the ``set_index`` method of the ``DataFrame``, which returns a multiply indexed ``DataFrame``:

In [ ]:
pop_df = pop_flat.set_index(['state', 'year'])
pop_df

In [ ]:
pop_df.rename_axis([None, None])

In [ ]:
asdf = pop_df.rename_axis([None, None]).unstack()
asdf

In [ ]:
asdf.columns

In [ ]:
asdf["area"] = 999
asdf

In [ ]:
asdf.columns

In [ ]:
print(type(asdf["area"]))
asdf["area"]

In [ ]:
print(type(asdf["population"]))
asdf["population"]

In [ ]:
pop_flat

In [ ]:
pop_df2 = pop_flat.set_index('state').rename_axis(None)
pop_df2

In [ ]:
pop_df

In [ ]:
pop_df.reset_index()

# Reading Series and DataFrames

In [ ]:
%%bash
head Pokemon.csv

In [ ]:
df = pd.read_csv("Pokemon.csv")

Imagine someboy gave you a random dataset. You don't know any of its contents. What are the first steps you do?

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df["Type 1"].value_counts()

In [ ]:
df["Legendary"].value_counts()

In [ ]:
#when working with csvs you really want to make sure if you want the first column as index-column!
df = pd.read_csv("Pokemon.csv", index_col=0)
df.tail()

In [ ]:
df.reset_index().tail()

In [ ]:
df.reset_index().drop_duplicates(subset="#").tail()

In [ ]:
df = df[df['Name'] != 'Volcanion']
df.tail()

In [ ]:
no_duplicates = df.reset_index().drop_duplicates(subset="#").reset_index().drop("index", axis=1)  
no_duplicates.tail()

In [ ]:
no_duplicates.set_index("#").to_csv('Pokemon_no_duplicates.csv')
#no_duplicates.to_excel('Pokemon_no_duplicates.xlsx', sheet_name='Sheet1')

In [ ]:
%%bash
head Pokemon_no_duplicates.csv

In [ ]:
gen_one = no_duplicates[no_duplicates["Generation"] == 1].set_index("#")
gen_one.tail()

In [ ]:
first_gen_dict = gen_one["Name"].to_dict()

In [ ]:
[str(key)+" : "+str(val) for index, (key, val) in enumerate(first_gen_dict.items()) if index < 9]

**Documentation!**

There are really really many arguments for this function, suiting all of your needs!

https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.read_csv.html

# Ufuncs and Aggregation

## Aggregation in Pandas

Aggregations are functions, where one or more dimensions of data are collapsed onto a single value, like the `max`, `sum` or `mean`- functions.

Stat-operations generally *exclude* missing data.

### For Series

In [ ]:
a = np.arange(7)
ser = pd.Series(a**2, index=a)
ser

In [ ]:
ser.sum()
#mean(), median(), min(), max(), ...

### For DataFrames

In [ ]:
df = pd.DataFrame({'A': a**2,
                   'B': a**3})
df

In [ ]:
df.mean()

In [ ]:
df.mean(axis=0)

In [ ]:
df.mean(axis='rows')

In [ ]:
df.mean(axis=1)

The following table summarizes some other built-in Pandas aggregations:

| Aggregation              | Description                     |
|--------------------------|---------------------------------|
| ``count()``              | Total number of items           |
| ``first()``, ``last()``  | First and last item             |
| ``mean()``, ``median()`` | Mean and median                 |
| ``min()``, ``max()``     | Minimum and maximum             |
| ``std()``, ``var()``     | Standard deviation and variance |
| ``mad()``                | Mean absolute deviation         |
| ``prod()``               | Product of all items            |
| ``sum()``                | Sum of all items                |

These are all methods of ``DataFrame`` and ``Series`` objects.

## Ufuncs


We know Ufuncs already from Numpy: It are vectorized functions that change all values of an array simultaneously. 

Pandas does the same, with a nice twist: for unary operations like negation and trigonometric functions, these ufuncs will *preserve index and column labels* in the output, and for binary operations such as addition and multiplication, Pandas will automatically *align indices* when passing the objects to the ufunc.


This means that keeping the context of data and combining data from different sources –both potentially error-prone tasks with raw NumPy arrays– become essentially foolproof ones with Pandas.

In [ ]:
rng = np.random.RandomState(0)
df = pd.DataFrame(rng.randint(0, 10, (3, 4)),
                  columns=['A', 'B', 'C', 'D'])
df

In [ ]:
np.exp(df)

### UFuncs: Index Alignment

For binary operations on two ``Series`` or ``DataFrame`` objects, Pandas will align indices in the process of performing the operation.
This is very convenient when working with incomplete data.

In [ ]:
area = pd.Series({'Alaska': 1723337, 'Texas': 695662,
                  'California': 423967}, name='area')
population = pd.Series({'California': 38332521, 'Texas': 26448193,
                        'New York': 19651127}, name='population')
area

In [ ]:
population

In [ ]:
area / population

In [ ]:
"divide" in dir(pd.DataFrame)

In [ ]:
popdens = area.divide(population, fill_value=0)
popdens

In [ ]:
popdens = popdens.replace([np.inf, -np.inf], np.nan)
popdens.dropna()

In [ ]:
A = pd.DataFrame(rng.randint(0, 20, (2, 2)),
                 columns=list('AB'))
A

In [ ]:
B = pd.DataFrame(rng.randint(0, 20, (3, 3)),
                 columns=list('ABC'))
B

In [ ]:
A+B

In [ ]:
A.add(B, fill_value=0)

### More Index-Alignment

In [ ]:
df = pd.DataFrame({'a': np.random.randint(3, size=10)}, index=np.arange(1, 20, 2))
df

Let's add a new column to this DataFrame!

In [ ]:
tmp = pd.Series([0]*len(df.index))
tmp

In [ ]:
#df['new'] = tmp   #changes the original one
df.assign(new=tmp) #creates a copy

In [ ]:
old_aligned, new_aligned = df.align(tmp, axis=0)
old_aligned

In [ ]:
new_aligned

In [ ]:
old_aligned.assign(new=new_aligned)

In [ ]:
tmp = pd.Series([0]*len(df.index), index=df.index)
tmp

In [ ]:
df['new'] = tmp
df

## .agg()

If you want to apply more than one operation (ufunc/aggregation), use `.agg()`:

In [ ]:
df = pd.DataFrame([[1, 2, 3],
                   [4, 5, 6],
                   [7, 8, 9],
                   [np.nan, np.nan, np.nan]],
                  columns=['A', 'B', 'C'])
df

In [ ]:
df.agg(['sum', 'min'])

In [ ]:
#you can also use different aggregations for different columns: 
df.agg({'A' : ['sum', 'min'], 'B' : ['min', 'max']})

## apply()

While some ufuncs (like cumsum or exp) are pre-defined by pandas, the method `apply` can be used to run an arbitrary function on all elements of a Series or DataFrame.

In [ ]:
a = np.arange(7)
df = pd.DataFrame({'A': a**2,
                   'B': a**3})
df

In [ ]:
df.cumsum()

In [ ]:
df.apply(np.cumsum)

In [ ]:
df["A_cumsum"] = df.cumsum()["A"]
df["B_cumsum"] = df.apply(np.cumsum)["B"]
df

Using Lambda-functions, we can combine `apply` with arbitrary functions. Note that the argument of the function is always an entire column of the dataset.

In [ ]:
df.apply(lambda x: print(x, end='\n\n'))

In [ ]:
df

In [ ]:
df['A'] + 1

In [ ]:
df.apply(lambda x: x+1)

In [ ]:
def my_more_complex_func(ser):
    res = []
    for elem in ser:
        print(elem if elem > 16 else -elem)
        res.append(elem if elem > 16 else -elem)
    return res

In [ ]:
df.apply(my_more_complex_func)

In [ ]:
df

In [ ]:
df.apply(lambda x: x.max() - x.min())

Note that `apply` works for both DataFrames and Series!

In [ ]:
df["A"].apply(lambda x: print(x))

In [ ]:
df["A_normed"] = df["A"].apply(lambda x: x/df["A"].max())
df

We can even use dictionaries with the apply-function!

In [ ]:
z_moves = {"Normal": "Breakneck Blitz", "Fighting": "All-Out Pummeling", "Flying": "Supersonic Skystrike", "Poison": "Acid Downpour", "Ground": "Tectonic Rage", "Rock": "Continental Crush", "Bug": "Savage Spin-Out", "Ghost": "Never-Ending Nightmare",
"Steel": "Corkscrew Crash", "Fire": "Inferno Overdrive", "Water": "Hydro Vortex", "Grass": "Bloom Doom", "Electric": "Gigavolt Havoc", "Psychic": "Shattered Psyche", "Ice": "Subzero Slammer", "Dragon": "Devastating Drake", "Dark": "Black Hole Eclipse", "Fairy": "Twinkle Tackle"}
df = pd.read_csv("Pokemon.csv")
df.head()

In [ ]:
df["Z-Move"] = df["Type 1"].apply(lambda x:z_moves[x])
df.head()

Using `apply`, we can also convert a list of Series into a DataFrame, by making the individual columns to Series:

In [ ]:
s = pd.Series([ ['Red', 'Green', 'White'], ['Red', 'Black'], ['Yellow']]) 
print(type(s))
s

In [ ]:
pd.Series([1, 2, 3])

In [ ]:
df = s.apply(pd.Series)
print(type(df))
df

**Note on speed:**

According to ([1]), `apply()` is twice as fast as looping through a df's `iterrows()`, and 8 times as fast as looping over python lists.

Note however, that while `apply()` is much faster at looping over the rows of your DataFrame/Series (by taking advantage of a number of internal optimizations, such as using iterators in Cython), it still inherently loops through rows. Whatever you're applying, you're still executing it once for every row. So, wherever you can use make use of vectorized Ufuncs, do so - that is far more optimized and parallelized - for ([1]) exchanging the haversine distance formula with it's vectorized counterpart led to a 50-fold-decrease in time!

\(1): https://engineering.upside.com/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6


[1]: https://engineering.upside.com/a-beginners-guide-to-optimizing-pandas-code-for-speed-c09ef2c6a4d6 

# Group-By

## Split-Apply-Combine

While simple operations are already pre-defined by pandas, custom aggregations and operations can be performed via **group-by**. The group-by operation can be described as having the following steps:

* **Splitting** the data into groups based on some criteria (breaking up and grouping depending on the value of a key)
* **Applying** a function to each group independently (aggregation, transformation, filtering, ...)
* **Combining** the results into a data structure

A typical example, for where the *apply* is a summerization aggregation, is illustrated here:

![](split-apply-combine.png)

In [ ]:
tmp = np.array([list("ABCABC"), np.arange(1,7)]).T
tmp

In [ ]:
df = pd.DataFrame(tmp, columns=["key", "data"])
df["data"] = pd.to_numeric(df["data"])
df

In [ ]:
df.groupby("key")

Note that what is returned is not a set of `DataFrames`, but a `DataFrameGroupBy` object. This object is where the magic is: you can think of it as a special view of the `DataFrames`, which is poised to dig into the groups but does no actual computation until the aggregation is applied. This "lazy evaluation" approach means that common aggregates can be implemented very efficiently in a way that is almost transparent to the user.

To produce a result, we can apply an aggregate to this `DataFrameGroupBy` object, which will perform the appropriate apply/combine steps to produce the desired result:

In [ ]:
df.groupby("key").sum().reset_index()

In [ ]:
df.groupby("key")["data"].sum()
# we can do column indexing just like on a normal DataFrame

### Iteration over groups

The ``GroupBy`` object supports direct iteration over the groups, returning each group as a ``Series`` or ``DataFrame``:

In [ ]:
df

In [ ]:
for (key, _) in df.groupby("key"):
    print(key)
    
print()
for (_, group) in df.groupby("key"):
    print(group, "\n")

In [ ]:
pkm = pd.read_csv('Pokemon.csv')
pkm.groupby('Generation')['Total'].mean()

### Dispatch methods

Any method not explicitly implemented by the ``GroupBy`` object will be passed through and called on the groups, whether they are ``DataFrame`` or ``Series`` objects.
For example, you can use the ``describe()`` method of ``DataFrame``s to perform a set of aggregations that describe each group in the data:

In [ ]:
df.describe()

In [ ]:
df.groupby("key").describe()

In [ ]:
df = pd.read_csv("Pokemon_no_duplicates.csv", index_col=0)
df.groupby('Generation')["Name"].nunique()